In [1]:
import re, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio.features import shapes, sieve
from shapely.geometry import shape as to_shape

from eudr import rasterstream as rs
from eudr.boundaries import utm_epsg_for
from eudr.config import load_config, run_dirs, slugify
from eudr.manifest import write_manifest

cfg = load_config(ROOT)
dirs = run_dirs(cfg)

FD = cfg["forest_data"]
H = FD["hansen"]
THR = H["treecover_threshold"]                                   # canopy % threshold
CUT_CODE = int(cfg["eudr"]["cutoff_date"][:4]) - 2000            # 20: loss codes 21+ are after the cut-off
LAST_CODE = int(re.search(r"GFC-(\d{4})", H["version"]).group(1)) - 2000
MIN_PATCH_HA = cfg.get("analysis", {}).get("min_patch_ha", 0.2)

CLASS_LABELS = {1: "Loss after 2020 inside 2020 forest (high concern)",
                2: "Loss after 2020 outside 2020 forest (review: tree crops or shade?)",
                3: "Loss before cut-off (2001-2020)"}
WC_NAMES = {10: "tree_cover", 20: "shrubland", 30: "grassland", 40: "cropland", 50: "built_up",
            60: "bare", 70: "snow_ice", 80: "water", 90: "herbaceous_wetland",
            95: "mangroves", 100: "moss_lichen"}

print("Hansen version:", H["version"], "| cut-off code:", CUT_CODE, "| last loss year:", 2000 + LAST_CODE)

Hansen version: GFC-2025-v1.13 | cut-off code: 20 | last loss year: 2025


In [7]:
gpkg = dirs["gis"] / "study_area.gpkg"
districts = gpd.read_file(gpkg, layer="ref_districts")
subcounties = gpd.read_file(gpkg, layer="ref_subcounties")
print(districts[["district", "area_ha", "utm_epsg"]])

  district    area_ha  utm_epsg
0   Buikwe  145203.90     32636
1  Kyotera  244803.31     32736


Helper Function

In [8]:
def vectorise_patches(cls, tr, crs, subs, dname, utm, pix_ha):
    """Turn loss classes 1 and 2 into polygons (small speckle removed) for QGIS."""
    keep = np.where(np.isin(cls, (1, 2)), cls, 0).astype("uint8")
    min_px = max(1, int(round(MIN_PATCH_HA / pix_ha)))
    keep = sieve(keep, size=min_px, connectivity=8)
    feats = [(to_shape(g), int(v)) for g, v in shapes(keep, mask=keep > 0, transform=tr)]
    cols = ["loss_class", "class_label", "area_ha", "district", "subcounty", "geometry"]
    if not feats:
        return gpd.GeoDataFrame(columns=cols, geometry="geometry", crs=crs)
    gdf = gpd.GeoDataFrame({"loss_class": [v for _, v in feats]},
                           geometry=[g for g, _ in feats], crs=crs)
    gdf["area_ha"] = (gdf.to_crs(utm).area / 10_000).round(3)
    gdf = gdf[gdf["area_ha"] >= MIN_PATCH_HA].copy()
    gdf["class_label"] = gdf["loss_class"].map(CLASS_LABELS)
    gdf["district"] = dname
    pts = gdf.copy()
    pts["geometry"] = gdf.representative_point()
    j = gpd.sjoin(pts[["geometry"]], subs[["subcounty", "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated()]
    gdf["subcounty"] = j.reindex(gdf.index)["subcounty"]
    return gdf[cols]


def make_map(cls, forest2020, tr, subs, dname, path):
    """Forest and loss map for one district."""
    rgb = np.full(cls.shape + (3,), 245, dtype="uint8")                    # background
    rgb[forest2020] = (120, 170, 120)                                      # forest in 2020
    rgb[cls == 3] = (240, 170, 80)                                         # loss before cut-off
    rgb[cls == 2] = (250, 220, 60)                                         # loss after, outside forest
    rgb[cls == 1] = (200, 30, 30)                                          # loss after, inside forest
    rgb[cls == 255] = (255, 255, 255)                                      # outside district
    h, w = cls.shape
    extent = (tr.c, tr.c + w * tr.a, tr.f + h * tr.e, tr.f)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb, extent=extent)
    subs.boundary.plot(ax=ax, color="black", linewidth=0.4)
    ax.set_title(f"{dname}: forest 2020 and tree-cover loss")
    handles = [plt.Rectangle((0, 0), 1, 1, color=np.array(c) / 255)
               for c in [(120, 170, 120), (240, 170, 80), (250, 220, 60), (200, 30, 30)]]
    ax.legend(handles, ["Forest 2020", "Loss 2001-2020", "Loss after 2020, outside forest (review)",
                        "Loss after 2020, inside forest (high concern)"], loc="lower left", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)
    

In [17]:
import os, certifi, requests, time
from pathlib import Path

os.environ["CURL_CA_BUNDLE"] = certifi.where()   # restart the kernel if GDAL was already used

TILE_DIR = ROOT / "data" / "cache" / "tiles"

def download_once(url, outdir, retries=8, delay=3, chunk=1 << 20):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    dest = outdir / url.split("?")[0].rstrip("/").split("/")[-1]
    if dest.exists() and dest.stat().st_size > 0:
        return str(dest)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        try:
            have = part.stat().st_size if part.exists() else 0
            headers = {"Range": f"bytes={have}-"} if have else {}
            with requests.get(url, stream=True, timeout=(15, 60), headers=headers) as r:
                if r.status_code == 416:                  # already complete
                    part.replace(dest); return str(dest)
                r.raise_for_status()
                mode = "ab" if r.status_code == 206 else "wb"
                got = have if mode == "ab" else 0
                with open(part, mode) as f:
                    for block in r.iter_content(chunk):
                        f.write(block); got += len(block)
                        if got // (50 << 20) != (got - len(block)) // (50 << 20):
                            print(f"  {dest.name}: {got/1e6:.0f} MB", flush=True)
            part.replace(dest)
            return str(dest)
        except Exception as e:
            if attempt == retries: raise
            print(f"Download failed ({e.__class__.__name__}); retry {attempt}/{retries} in {delay}s")
            time.sleep(delay); delay = min(delay * 2, 60)



Stream, Clip and Analyze Each District

In [ ]:
stats_parts, annual_rows, patch_parts, used_sources = [], [], [], {}
baseline_used = None
for _, d in districts.iterrows():
    dname, geom = d["district"], d.geometry
    slug = slugify(dname)
    cache = dirs["cache"]
    subs = subcounties[subcounties["district"] == dname].copy().reset_index(drop=True)
    subs["zone_id"] = np.arange(1, len(subs) + 1)
    utm = int(d["utm_epsg"])
    bounds = rs.padded_bounds(geom.bounds)
    print(f"\n=== {dname}: tiles {rs.hansen_tile_ids(bounds)} ===")

    # 1. Hansen: download each tile once (cached), then clip to the district
    tc_urls = rs.hansen_urls("treecover2000", bounds, H["base_url"], H["version"])
    ly_urls = rs.hansen_urls("lossyear", bounds, H["base_url"], H["version"])
    tc_src = [download_once(u, TILE_DIR / "hansen") for u in tc_urls]
    ly_src = [download_once(u, TILE_DIR / "hansen") for u in ly_urls]
    tc, tr, crs = rs.get_or_stream(cache / f"{slug}_hansen_treecover2000_{H['version']}.tif", tc_src, bounds)
    ly, _, _ = rs.get_or_stream(cache / f"{slug}_hansen_lossyear_{H['version']}.tif", ly_src, bounds)
    assert tc.shape == ly.shape, "Hansen layers do not share a grid"
    shape = tc.shape
    used_sources[dname] = {"hansen_treecover2000": tc_urls, "hansen_lossyear": ly_urls}

    # 2. Clip mask, subcounty labels, pixel areas
    inside = rs.inside_mask(geom, shape, tr)
    labels = rs.label_grid(subs, "zone_id", shape, tr)
    area_row = rs.pixel_area_ha(tr, shape)
    w = rs.weights_2d(area_row, shape)
    n = len(subs)

    # 3. Hansen-derived forest baseline (proxy, not the EUDR reference map)
    forest2020 = (tc >= THR) & ~((ly >= 1) & (ly <= CUT_CODE))
    baseline_used = "Hansen proxy (NOT the EUDR reference map)"

    # 4. WorldCover  # CHANGED: download first
    wc_urls = rs.worldcover_sources(bounds, FD["worldcover"]["year"], FD["worldcover"]["stac_url"])
    wc_src = [download_once(u, TILE_DIR / "worldcover") for u in wc_urls]
    wc_raw, wtr, wcrs = rs.get_or_stream(cache / f"{slug}_worldcover_{FD['worldcover']['year']}.tif",
                                         wc_src, bounds, nodata=0)
    wc = rs.align_to(wc_raw, wtr, wcrs, shape, tr, crs, Resampling.mode, dtype="uint8")
    used_sources[dname]["worldcover"] = f"ESA WorldCover {FD['worldcover']['year']} (Planetary Computer)"

    # 5. Loss classes (unchanged)
    loss_any = (ly > 0) & (tc >= THR)
    loss_pre = loss_any & (ly <= CUT_CODE)
    loss_post = loss_any & (ly > CUT_CODE)
    cls = np.full(shape, 255, dtype="uint8")
    cls[inside] = 0
    cls[inside & loss_pre] = 3
    cls[inside & loss_post & ~forest2020] = 2
    cls[inside & loss_post & forest2020] = 1

    # 6. Subcounty statistics (unchanged)
    def zs(mask):
        return rs.zonal_area_ha(labels, mask & inside, w, n)

    table = pd.DataFrame({
        "zone_id": subs["zone_id"],
        "tree_cover_2000_ha": zs(tc >= THR),
        "forest_2020_ha": zs(forest2020),
        "loss_2001_2020_ha": zs(loss_pre),
        "loss_post_2020_ha": zs(loss_post),
        "loss_post_2020_in_forest_ha": zs(loss_post & forest2020),
        "loss_post_2020_outside_forest_ha": zs(loss_post & ~forest2020),
    })
    for code in np.unique(wc[inside]):
        if code in WC_NAMES:
            table[f"wc_{WC_NAMES[code]}_ha"] = zs(wc == code)
    subs_stats = subs.merge(table, on="zone_id")
    subs_stats["mapped_area_ha"] = zs(np.ones(shape, bool))
    subs_stats["forest_2020_pct"] = (subs_stats["forest_2020_ha"] / subs_stats["mapped_area_ha"] * 100).round(1)
    stats_parts.append(subs_stats)

    # 7. Annual loss (unchanged)
    for code in range(1, LAST_CODE + 1):
        m = inside & loss_any & (ly == code)
        annual_rows.append({"district": dname, "year": 2000 + code,
                            "loss_ha": float(w[m].sum()),
                            "loss_in_forest2020_ha": float(w[m & forest2020].sum())})

    # 8. Outputs (unchanged)
    rs.write_tif(dirs["rasters"] / f"{slug}_forest_2020.tif", forest2020.astype("uint8"), tr, crs, nodata=255)
    rs.write_tif(dirs["rasters"] / f"{slug}_loss_class.tif", cls, tr, crs, nodata=255)
    rs.write_tif(dirs["rasters"] / f"{slug}_worldcover_{FD['worldcover']['year']}.tif", wc, tr, crs, nodata=0)
    patch_parts.append(vectorise_patches(cls, tr, crs, subs, dname, utm, float(area_row.mean())))
    make_map(cls, forest2020, tr, subs, dname, dirs["maps"] / f"{slug}_forest_loss.png")
    print(f"{dname}: done ({n} subcounties, {int(loss_post[inside].sum())} post-2020 loss pixels)")

Combine, Summarize and Save the geopackage and excel formats

In [ ]:
import geopandas as gpd
import pandas as pd

stats = gpd.GeoDataFrame(pd.concat(stats_parts, ignore_index=True), crs=subcounties.crs)
annual = pd.DataFrame(annual_rows)

# patches already carry the raster CRS from vectorise_patches; don't override it
patches = pd.concat(patch_parts, ignore_index=True)
if not isinstance(patches, gpd.GeoDataFrame):
    patches = gpd.GeoDataFrame(patches, geometry="geometry", crs=stats.crs)
elif patches.crs is None:
    patches = patches.set_crs(stats.crs)

ha_cols = [c for c in stats.columns if c.endswith("_ha") and c != "area_ha"]
district_summary = stats.groupby("district")[ha_cols].sum().round(1)
district_summary["forest_2020_pct"] = (
    district_summary["forest_2020_ha"] / district_summary["mapped_area_ha"] * 100).round(1)

display(district_summary[["mapped_area_ha", "forest_2020_ha", "forest_2020_pct",
                          "loss_post_2020_in_forest_ha", "loss_post_2020_outside_forest_ha"]])

# GeoPackage for QGIS
out_gpkg = dirs["gis"] / "forest_analysis.gpkg"
if out_gpkg.exists():
    out_gpkg.unlink()
stats.to_file(out_gpkg, layer="forest_subcounty_stats", driver="GPKG")
if len(patches):
    patches.to_file(out_gpkg, layer="risk_loss_post2020_patches", driver="GPKG")
else:
    print("No post-2020 loss patches to write.")

# Excel report
notes = [
    f"2020 baseline used: {baseline_used}",
    f"Hansen {H['version']}, tree cover threshold {THR}%",
    "Loss class 1 = loss after 31 Dec 2020 inside 2020 forest",
    "Loss class 2 = loss after 31 Dec 2020 outside 2020 forest (review)",
    "Loss class 3 = loss 2001-2020 (before the cut-off)",
]
with pd.ExcelWriter(dirs["reports"] / "forest_stats.xlsx") as xl:
    district_summary.reset_index().to_excel(xl, sheet_name="district_summary", index=False)
    stats.drop(columns="geometry").round(2).to_excel(xl, sheet_name="subcounty_stats", index=False)
    annual.round(2).to_excel(xl, sheet_name="annual_loss", index=False)
    pd.DataFrame({"note": notes}).to_excel(xl, sheet_name="notes", index=False)

print("saved", out_gpkg)


districts: 2 ['Buikwe', 'Kyotera']
stats_parts: 0 | annual_rows: 0 | patch_parts: 0


Annual Tree Loss Chart

In [ ]:
fig, axes = plt.subplots(1, len(districts), figsize=(6 * len(districts), 4), sharey=True, squeeze=False)
for ax, (dname, grp) in zip(axes[0], annual.groupby("district")):
    outside = grp["loss_ha"] - grp["loss_in_forest2020_ha"]
    ax.bar(grp["year"], grp["loss_in_forest2020_ha"], label="Inside 2020 forest", color="#c81e1e")
    ax.bar (grp["year"], outside, bottom=grp["loss_in_forest2020_ha"], label="Outside 2020 forest",color = "#f0aa50")
    ax.axvline (2020.5, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{dname}: annual tree-cover loss (ha)")
    ax.set_xlabel("Year")
axes[0][0].set_ylabel("Hectares")
axes[0][0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(dirs["maps"] / "annual_loss.png", dpi=200)
plt.show()


Calculate Normalized Vegetation Index (NDVI)

In [ ]:
import dask
import numpy as np
import odc.stac
import planetary_computer
import pystac_client
import xarray as xr
from affine import Affine


def ndvi_composite(bounds, date_range: str, max_cloud: float, epsg: int,
                   resolution_m: float, stac_url: str, max_scenes: int = 40):
    """Stream Sentinel-2 L2A red (B04), NIR (B08) and the scene classification (SCL),
    mask clouds, compute NDVI per scene, and take the median.

    bounds : (west, south, east, north) in degrees
    epsg   : UTM zone to compute in (metres)
    Returns: ndvi (float32), n_obs (uint16), affine transform, list of scenes used
    """
    # retries + cloud-friendly GDAL settings for every read odc.stac makes
    odc.stac.configure_rio(cloud_defaults=True,
                           GDAL_HTTP_MAX_RETRY=5,
                           GDAL_HTTP_RETRY_DELAY=2)

    catalog = pystac_client.Client.open(stac_url, modifier=planetary_computer.sign_inplace)
    items = list(catalog.search(collections=["sentinel-2-l2a"], bbox=bounds, datetime=date_range,
                                query={"eo:cloud_cover": {"lt": max_cloud}}).items())
    if not items:
        raise ValueError("No Sentinel-2 scenes found. Widen date_range or raise max_cloud_cover.")
    items.sort(key=lambda it: it.properties.get("eo:cloud_cover", 100))
    items = items[:max_scenes]
    print(f"Using {len(items)} Sentinel-2 scenes (cloud cover "
          f"{items[0].properties['eo:cloud_cover']:.0f}% to {items[-1].properties['eo:cloud_cover']:.0f}%)")

    ds = odc.stac.load(items, bands=["B04", "B08", "SCL"], bbox=bounds,
                       crs=f"EPSG:{epsg}", resolution=resolution_m,
                       groupby="solar_day", resampling="nearest",
                       chunks={"x": 512, "y": 512},
                       fail_on_error=False)          # failed reads become gaps (0), not a crash

    # Scenes processed from about 25 Jan 2022 carry a +1000 offset in the digital numbers.
    offset = xr.where(ds["time"] >= np.datetime64("2022-01-25"), 1000.0, 0.0)
    red = ((ds["B04"].astype("float32") - offset) / 10000.0).clip(min=0)
    nir = ((ds["B08"].astype("float32") - offset) / 10000.0).clip(min=0)

    # keep clear pixels: SCL 4 vegetation, 5 bare soil, 6 water (drops cloud, shadow, cirrus)
    valid = (ds["B04"] > 0) & (ds["B08"] > 0) & ds["SCL"].isin([4, 5, 6]) & ((nir + red) > 0.01)
    ndvi = ((nir - red) / (nir + red)).where(valid).clip(-1, 1)

    ndvi = ndvi.chunk({"time": -1})                  # median needs one chunk along time
    ndvi_med = ndvi.median("time", skipna=True)
    n_obs = ndvi.notnull().sum("time")

    with dask.config.set(scheduler="threads", num_workers=4):   # fewer parallel connections
        ndvi_med, n_obs = dask.compute(ndvi_med, n_obs)         # one pass over the streamed data

    x, y = ds["x"].values, ds["y"].values
    rx, ry = float(x[1] - x[0]), float(y[1] - y[0])
    transform = Affine(rx, 0, float(x[0]) - rx / 2, 0, ry, float(y[0]) - ry / 2)

    scenes = [{"id": it.id, "date": str(it.datetime.date()),
               "cloud_cover": it.properties.get("eo:cloud_cover")} for it in items]
    return ndvi_med.values.astype("float32"), n_obs.values.astype("uint16"), transform, scenes

NDVI_calculations

In [18]:
from eudr.config import load_config
cfg = load_config(ROOT)          # re-read config.yaml from disk
FD = cfg["forest_data"]          # refresh the variable this cell uses
print("ndvi keys now:", list(FD["ndvi"]))
NV = FD["ndvi"]
date_range_str = NV.get("date_range", "2025-01-01/2025-12-31")
tag = date_range_str.replace("/", "_to_")
max_cloud = NV.get("max_cloud_cover", 10)
max_scenes = NV.get("max_scenes", 5)
resolution = NV.get("resolution_m", 10)
REFRESH_NDVI = False      

if not NV["enabled"]:
    print("NDVI skipped: set forest_data.ndvi.enabled: true in config.yaml, then re-run cell 1.")
else:
    ndvi_tables, ndvi_files, ndvi_scenes = [], {}, {}

    for _, d in districts.iterrows():
        dname, slug = d["district"], slugify(d["district"])
        epsg = int(d["utm_epsg"])
        bounds = rs.padded_bounds(d.geometry.bounds)
        ndvi_tif = dirs["rasters"] / f"{slug}_ndvi_median_{tag}.tif"
        nobs_tif = dirs["rasters"] / f"{slug}_ndvi_nobs_{tag}.tif"
        print(f"\n=== NDVI {dname} (UTM EPSG:{epsg}) ===")

        # District polygon and subcounties in the same UTM grid as the NDVI raster
        geom_utm = gpd.GeoSeries([d.geometry], crs=districts.crs).to_crs(epsg).iloc[0]
        subs = subcounties[subcounties["district"] == dname].copy().reset_index(drop=True)
        subs["zone_id"] = np.arange(1, len(subs) + 1)      # same order as in cell 4
        subs_utm = subs.to_crs(epsg)

        if ndvi_tif.exists() and not REFRESH_NDVI:
            ndvi, transform, _ = rs.read_tif(ndvi_tif)
            print("Loaded saved NDVI:", ndvi_tif.name)
        else:
            ndvi, n_obs, transform, scenes = rs.ndvi_composite(
                bounds, date_range_str, max_cloud, epsg,
                resolution, FD["worldcover"]["stac_url"], max_scenes)
            inside = rs.inside_mask(geom_utm, ndvi.shape, transform)
            ndvi = np.where(inside, ndvi, np.nan).astype("float32")
            n_obs = np.where(inside, n_obs, 0).astype("uint16")

            rs.write_tif(ndvi_tif, ndvi, transform, f"EPSG:{epsg}", nodata=np.nan)
            rs.write_tif(nobs_tif, n_obs, transform, f"EPSG:{epsg}")
            ndvi_scenes[dname] = scenes
            print("Saved:", ndvi_tif)
            print("Saved:", nobs_tif)
        ndvi_files[dname] = str(ndvi_tif)

        # NDVI statistics per subcounty
        labels = rs.label_grid(subs_utm, "zone_id", ndvi.shape, transform)
        subs["ndvi_mean"] = rs.zonal_mean(labels, ndvi, len(subs)).round(3)
        ndvi_tables.append(subs[["district", "subcounty", "ndvi_mean"]])

        valid_vals = ndvi[np.isfinite(ndvi)]
        print(f"NDVI range {valid_vals.min():.2f} to {valid_vals.max():.2f}, "
              f"mean {valid_vals.mean():.2f}, {np.isfinite(ndvi).mean() * 100:.0f}% of pixels have data")

    ndvi_table = pd.concat(ndvi_tables, ignore_index=True)
    ndvi_table.to_excel(dirs["reports"] / f"ndvi_stats_{tag}.xlsx", index=False)
    display(ndvi_table.sort_values("ndvi_mean").head(10))


ndvi keys now: ['enabled', 'data_range', 'max_cloud_cover', 'max_scenes', 'resolution_m']
NDVI skipped: set forest_data.ndvi.enabled: true in config.yaml, then re-run cell 1.


Manifest

In [53]:
write_manifest(dirs["manifests"], "02b_ndvi", {
        "date_range": date_range_str, "max_cloud_cover": max_cloud,
        "max_scenes": max_scenes, "resolution_m": resolution,
        "files": ndvi_files, "scenes_used": ndvi_scenes,
})

WindowsPath('C:/Users/shyak/Downloads/eudr-coffee-uganda/outputs/buikwe_kyotera/manifests/02b_ndvi.json')